In [2]:
import os, time, json
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

ENV_FILE = find_dotenv(usecwd=True)          # sucht .env ab dem Notebook-Ordner (labs/.env)
print("Konfiguration aus:", ENV_FILE or "keine .env gefunden, Defaults (OpenAI, gpt-4.1)")
load_dotenv(ENV_FILE, override=True)   # override: CrewAI lädt beim Import schon eine .env aus einem Elternordner
os.environ.setdefault("CREWAI_DISABLE_TELEMETRY", "true")
os.environ.setdefault("OTEL_SDK_DISABLED", "true")

LLM_BASE_URL = os.environ.get("LLM_BASE_URL", "https://api.openai.com/v1")
LLM_API_KEY = os.environ.get("LLM_API_KEY", "")
LLM_MODEL = os.environ.get("LLM_MODEL", "gpt-4.1")

from pydantic import BaseModel
from crewai import Agent, Task, Crew, Process, LLM, TaskOutput

DATA = Path("data")
diff_text = (DATA / "diff_rabatt.patch").read_text(encoding="utf-8")
print(f"Diff geladen: {len(diff_text.splitlines())} Zeilen")
print(diff_text[:600])

llm = LLM(model=f"openai/{LLM_MODEL}", base_url=LLM_BASE_URL, api_key=LLM_API_KEY, temperature=0.1)
print("Modell:", llm.model, "| Endpunkt:", LLM_BASE_URL)
print("LLM antwortet:", llm.call("Antworte nur mit OK.")[:40])

Konfiguration aus: /Users/grigo/gits/seminars/allSkills/ki-agenten-crewai/src_live/.env
Diff geladen: 43 Zeilen
diff --git a/shop/rabatt.py b/shop/rabatt.py
index 3f1c2a9..8be47d1 100644
--- a/shop/rabatt.py
+++ b/shop/rabatt.py
@@ -1,12 +1,37 @@
 """Preisberechnung für den Webshop."""
 
 from decimal import Decimal
+
+# Rabattstaffel: ab Bestellwert (inklusive) gilt der Prozentsatz.
+# 0 bis 99,99 EUR: 0 %, 100 bis 499,99 EUR: 5 %, ab 500 EUR: 10 %
+STAFFEL = [
+    (Decimal("500"), Decimal("0.10")),
+    (Decimal("100"), Decimal("0.05")),
+]
+
+
+def rabattsatz(bestellwert: Decimal) -> Decimal:
+    """Liefert den Rabattsatz für einen Bestellwert."""
+    if bestellwert < 0:
+        raise ValueError(
Modell: gpt-4.1 | Endpunkt: https://api.openai.com/v1
LLM antwortet: OK.
